## **Bank Nifty Predictability Under Regulatory Expiry-Structure Reform**

### **Notebook 03: Feature Engineering and Cyclical Expiry Distance**

[![Numpy](https://img.shields.io/badge/Numpy-Latest-blue)](https://numpy.org/) [![Scikit-Learn](https://img.shields.io/badge/Scikit--Learn-Latest-blue)](https://scikit-learn.org/)

---

**Scope:** Construct baseline technical analysis indicators and cyclical distance-to-expiry features for derivatives-structure aware forecasting.

---


### **Pipeline Context**

This is a sequential 9-notebook experimental workflow:

1. **01_data_collection.ipynb**
2. **02_data_cleaning_eda.ipynb**
3. **03_feature_engineering.ipynb**
4. **04_volatility_regime_detection.ipynb**
5. **05_baseline_models.ipynb**
6. **06_proposed_model.ipynb**
7. **07_ablation_study.ipynb**
8. **08_evaluation_and_diagnostics.ipynb**
9. **09_figures_for_paper.ipynb**

---
**Current Location:** 3. **03_feature_engineering.ipynb** <- *You are here*

---


## 1. Introduction

### 1.1 Purpose of This Notebook
Most quantitative forecasting literature treats stock index series as simple stationary time series, feeding them generic indicators like RSI, MACD, and Bollinger Bands. In this notebook, we engineer these standard features as a baseline, but also construct **derivatives-structure aware features**:
1. **Days-to-Expiry**: Count of trading days remaining in the monthly options cycle.
2. **Cyclical Expiry Distance**: Sine and Cosine transformations of the days-to-expiry distance. This cyclical mapping ensures that the model treats days immediately following expiry and days leading up to it as conceptually adjacent.
3. **Expiry-Week Indicator**: A binary flag identifying the final week (last 5 trading days) of the monthly options cycle.

## 2. Environment Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

project_dir = Path("e:/BN_Structure_Reforms/banknifty-expiry-volatility-ml")
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

from src import config
from src.features import add_all_features

## 3. Feature Generation Pipeline

In [2]:
df_clean = pd.read_parquet(config.INTERIM_DATA_DIR / "banknifty_clean.parquet")
df_expiry = pd.read_csv(config.RAW_DATA_DIR / "nse_expiry_calendar.csv")

# Compute both baseline technical and proposed cyclical expiry features
df_features = add_all_features(df_clean, df_expiry['ActualExpiry'])
df_features = df_features.dropna()

print(f"Total features generated: {df_features.shape[1]}")
print("Feature list:", df_features.columns.tolist())

Total features generated: 22
Feature list: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Return', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Mid', 'BB_Upper', 'BB_Lower', 'BB_PctB', 'BB_Width', 'ATR', 'Rolling_Vol', 'Days_To_Expiry', 'Expiry_Week', 'Expiry_Dist_Sin', 'Expiry_Dist_Cos']


## 4. Cyclical Expiry Distance Verification

### 4.1 Inspecting Distance Transformations

In [3]:
inspect_cols = ['Date', 'Close', 'Days_To_Expiry', 'Expiry_Week', 'Expiry_Dist_Sin', 'Expiry_Dist_Cos']
df_features[inspect_cols].head(15)

,Date,Close,Days_To_Expiry,Expiry_Week,Expiry_Dist_Sin,Expiry_Dist_Cos
19,2023-06-28,44327.800781,0,1,0.000000e+00,1.000000e+00
20,2023-06-30,44747.351562,19,0,-3.090170e-01,9.510565e-01
21,2023-07-03,45158.101562,18,0,-5.877853e-01,8.090170e-01
22,2023-07-04,45301.449219,17,0,-8.090170e-01,5.877853e-01
23,2023-07-05,45151.800781,16,0,-9.510565e-01,3.090170e-01
24,2023-07-06,45339.898438,15,0,-1.000000e+00,-1.836970e-16
25,2023-07-07,44925.000000,14,0,-9.510565e-01,-3.090170e-01
26,2023-07-10,44860.851562,13,0,-8.090170e-01,-5.877853e-01
27,2023-07-11,44745.050781,12,0,-5.877853e-01,-8.090170e-01
28,2023-07-12,44639.449219,11,0,-3.090170e-01,-9.510565e-01


## 5. Export Feature Set

In [4]:
df_features.to_parquet(config.INTERIM_DATA_DIR / "banknifty_features.parquet", index=False)
print(f"Saved dataset with features to {config.INTERIM_DATA_DIR / 'banknifty_features.parquet'}")

Saved dataset with features to E:\BN_Structure_Reforms\banknifty-expiry-volatility-ml\data\interim\banknifty_features.parquet
